In [3]:
import pandas as pd

df = pd.read_csv('email_spam_indo.csv')
df.head()

,Kategori,Pesan
0,spam,Secara alami tak tertahankan identitas perusah...
1,spam,Fanny Gunslinger Perdagangan Saham adalah Merr...
2,spam,Rumah -rumah baru yang luar biasa menjadi muda...
3,spam,4 Permintaan Khusus Pencetakan Warna Informasi...
4,spam,"Jangan punya uang, dapatkan CD perangkat lunak..."


In [5]:
df = df.rename(columns={
    'Pesan': 'text',
    'Kategori': 'label'
})

In [6]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'(dikeret oleh|ditahan oleh|ect pada|subjek:).*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b\w+\s+(com|net|org|co|id)\b', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'^(re|fw|fwd):[^.?!\n]*', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(preprocess)
df[['text', 'clean_text']].head()

,text,clean_text
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...


In [7]:
NORMALIZATION_DICT = {
    # 🔤 Singkatan umum
    "gk": "tidak",
    "d": "di",
    "ga": "tidak",
    "gak": "tidak",
    "nggak": "tidak",
    "tdk": "tidak",
    "dr": "dari",
    "dgn": "dengan",
    "utk": "untuk",
    "yg": "yang",
    "jg": "juga",
    "krn": "karena",
    "blm": "belum",
    "sdh": "sudah",
    "aja": "saja",
    "trs": "terus",
    "bgt": "banget",
    "dlm": "dalam",

    # 💰 Kata khas spam / promo
    "rp": "rupiah",
    "jt": "juta",
    "rb": "ribu",
    "cashback": "cashback",
    "promo": "promosi",
    "diskon": "diskon",
    "gratis": "gratis",
    "hadiah": "hadiah",
    "menang": "menang",
    "undian": "undian",

    # 📱 Variasi kata penting
    "tlp": "telepon",
    "telp": "telepon",
    "hp": "handphone",
    "no": "nomor",
    "rek": "rekening",
    "an": "atas nama",

    # ⚡ Variasi penulisan spam
    "selamat!!!": "selamat",
    "selamat!!": "selamat",
    "selamat!": "selamat",
    "trnsfer": "transfer",
    "trf": "transfer",

    # 🧩 Variasi angka ke huruf (leetspeak)
    "4nda": "anda",
    "4pa": "apa",
    "1ni": "ini",
    "k4mu": "kamu",

    # 🪤 Kata clickbait spam
    "klik": "klik",
    "segera": "segera",
    "buruan": "cepat",
    "cepat": "cepat",
    "terbatas": "terbatas",
    "khusus": "khusus",

    # 🎯 Kata inti spam
    "free": "gratis",
    "win": "menang",
    "winner": "pemenang",
    "prize": "hadiah",
    "bonus": "bonus",
    "gift": "hadiah",
    "reward": "hadiah",

    # 💰 Finansial / uang
    "cash": "uang",
    "money": "uang",
    "credit": "kredit",
    "loan": "pinjaman",
    "bank": "bank",
    "transfer": "transfer",
    "account": "akun",
    "sspecial": "spesial",
    "balance": "saldo",

    # 📢 Promosi / marketing
    "promo": "promosi",
    "offer": "penawaran",
    "deal": "penawaran",
    "discount": "diskon",
    "sale": "diskon",
    "limited": "terbatas",
    "exclusive": "eksklusif",

    # ⚡ Call to action (penting banget buat spam)
    "click": "klik",
    "claim": "klaim",
    "buy": "beli",
    "order": "pesan",
    "register": "daftar",
    "subscribe": "langganan",
    "join": "gabung",
    "verify": "verifikasi",
    "confirm": "konfirmasi",

    # ⏰ urgency
    "urgent": "segera",
    "now": "sekarang",
    "today": "hari ini",
    "instant": "instan",

    # 🖥️ teknologi / produk
    "software": "perangkat lunak",
    "system": "sistem",
    "update": "pembaruan",
    "download": "unduh",
    "install": "instal",

    # 📧 email umum
    "hello": "halo",
    "dear": "halo",
    "sir": "bapak",
    "madam": "ibu",

    # 🧩 lain-lain yang sering muncul
    "service": "layanan",
    "customer": "pelanggan",
    "support": "dukungan",
    "information": "informasi",
    "message": "pesan"
}
def normalize_text(text):
    words = text.split()
    normalized_words = [NORMALIZATION_DICT.get(word, word) for word in words]
    return ' '.join(normalized_words)

df['normalized_text'] = df['clean_text'].apply(normalize_text)
df[['text', 'clean_text','normalized_text']].head()


,text,clean_text,normalized_text
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...,fanny gunslinger perdagangan saham adalah merr...
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...,rumah rumah baru yang luar biasa menjadi mudah...
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...,permintaan khusus pencetakan warna informasi t...
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...,jangan punya uang dapatkan cd perangkat lunak ...


In [8]:
!pip install Sastrawi
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 2.8 MB/s eta 0:00:00


In [9]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# ambil stopword default
factory = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

# fungsi hapus stopword
def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stopwords])

# apply ke data
df['stopwords'] = df['normalized_text'].apply(remove_stopwords)

# lihat hasil
df[['text', 'normalized_text', 'stopwords']].head()

,text,normalized_text,stopwords
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...,alami tak tertahankan identitas perusahaan san...
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...,fanny gunslinger perdagangan saham merrill muz...
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...,rumah rumah baru luar biasa menjadi mudah menu...
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...,permintaan khusus pencetakan warna informasi t...
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...,jangan punya uang dapatkan cd perangkat lunak ...


In [10]:
# tokenization
df['tokens'] = df['stopwords'].apply(lambda x: x.split())

# lihat hasil
df[['text', 'clean_text', 'stopwords', 'tokens']].head()

,text,clean_text,stopwords,tokens
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...,alami tak tertahankan identitas perusahaan san...,"[alami, tak, tertahankan, identitas, perusahaa..."
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...,fanny gunslinger perdagangan saham merrill muz...,"[fanny, gunslinger, perdagangan, saham, merril..."
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...,rumah rumah baru luar biasa menjadi mudah menu...,"[rumah, rumah, baru, luar, biasa, menjadi, mud..."
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...,permintaan khusus pencetakan warna informasi t...,"[permintaan, khusus, pencetakan, warna, inform..."
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...,jangan punya uang dapatkan cd perangkat lunak ...,"[jangan, punya, uang, dapatkan, cd, perangkat,..."


In [11]:
def calc_TF(document):
    TF_dict = {}

    for term in document:
        if term in TF_dict:
            TF_dict[term] += 1
        else:
            TF_dict[term] = 1

    # normalisasi
    for term in TF_dict:
        TF_dict[term] = TF_dict[term] / len(document)

    return TF_dict

df["TF_dict"] = df['tokens'].apply(calc_TF)

In [12]:
def calc_DF(tfDict_series):
    count_DF = {}

    for document in tfDict_series:
        for term in document:
            if term in count_DF:
                count_DF[term] += 1
            else:
                count_DF[term] = 1

    return count_DF

DF = calc_DF(df["TF_dict"])

In [13]:
import numpy as np

n_document = len(df)

def calc_IDF(n_document, DF):
    IDF_Dict = {}

    for term in DF:
        IDF_Dict[term] = np.log(n_document / (DF[term] + 1))

    return IDF_Dict

IDF = calc_IDF(n_document, DF)

In [14]:
def calc_TF_IDF(TF):
    TF_IDF_Dict = {}

    for term in TF:
        TF_IDF_Dict[term] = TF[term] * IDF[term]

    return TF_IDF_Dict

df["TF_IDF_dict"] = df["TF_dict"].apply(calc_TF_IDF)

In [15]:
index = 10  # bebas pilih index

print('%20s' % "term", "\t", '%10s' % "TF", "\t", '%20s' % "TF-IDF\n")

for key in df["TF_IDF_dict"][index]:
    print('%20s' % key,
          "\t", df["TF_dict"][index][key],
          "\t", df["TF_IDF_dict"][index][key])

                term 	         TF 	              TF-IDF

                 las 	 0.056338028169014086 	 0.326624019940426
               vegas 	 0.056338028169014086 	 0.33414691529955404
                high 	 0.028169014084507043 	 0.16707345764977702
                rise 	 0.014084507042253521 	 0.10118127767693595
                boom 	 0.028169014084507043 	 0.163312009970213
               cepat 	 0.014084507042253521 	 0.032356169994875714
             menjadi 	 0.014084507042253521 	 0.025118987684186793
                kota 	 0.014084507042253521 	 0.05070917995220201
        metropolitan 	 0.014084507042253521 	 0.10118127767693595
               utama 	 0.014084507042253521 	 0.04316666958098564
              menara 	 0.014084507042253521 	 0.08827577441110278
          bertingkat 	 0.014084507042253521 	 0.09547050150639842
              tinggi 	 0.014084507042253521 	 0.03869181944646936
                baru 	 0.028169014084507043 	 0.04833269222330482
          diharapkan 